# The loss landscape of a training run

This notebook trains a small transformer and draws, while it trains, the loss surface around its current weights in 3-D: the loss of a fixed batch on a 41 by 41 grid over a plane through the current weights, the path that the optimizer took in that plane as a line, and the current point as a red marker, inside a box with a floor grid. Every 500 steps the plane and the surface are computed again on the GPU. By default the plane is the one in which the weights have moved most so far, so the picture shows how the optimizer moves across the surface. A control switches it to two random directions.

It needs the `caliper` package with the extra `caliper[notebook]`, PyTorch, and a notebook front end such as JupyterLab or VS Code. On an Apple Silicon Mac the model trains on the GPU through MPS. The model, the data and the training loop are those of the tutorial [Watch a training run in a notebook](../../docs/tutorials/watch-a-training-run-in-a-notebook.md), repeated here so that the notebook runs alone. The file `landscape.py` beside this notebook computes the landscape.

`caliper.serve()` starts a server for this notebook's views on 127.0.0.1 and prints a link that carries a session token. Open the link in a browser tab to see the same scene and controls there. The server accepts connections only from this machine and only with the token.

In [ ]:
import math
import os
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

import caliper
from landscape import Landscape

server = caliper.serve()

## The data

Each example is a sequence of 24 random tokens. One position holds a marker token, and the token right after the marker decides the class, one of six. The six class tokens also appear at random elsewhere in the sequence, so the model has to find the marker to get the class right. The data is generated on the device at every step, so there is nothing to download.

The first eight sequences of every batch are the tutorial's fixed probe sequences. They are kept so that this run is the same run as the tutorial's.

`STEPS` sets the length of the run. Set the environment variable `CALIPER_NOTEBOOK_STEPS` to shorten it, for example to 200 for a quick test.

In [ ]:
STEPS = int(os.environ.get("CALIPER_NOTEBOOK_STEPS", "8000"))
BATCH = 256
VOCAB, LENGTH, CLASSES, PROBES = 32, 24, 6, 8
START, MARKER, FIRST_CLASS_TOKEN = 0, 1, 2     # the class tokens are 2 to 7

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
    print("MPS is not available, so the model trains on the CPU, more slowly.")

torch.manual_seed(0)


def make_batch(n):
    x = torch.randint(2, VOCAB, (n, LENGTH), device=device)
    y = torch.randint(0, CLASSES, (n,), device=device)
    where = torch.randint(1, LENGTH - 1, (n,), device=device)
    rows = torch.arange(n, device=device)
    x[rows, where] = MARKER
    x[rows, where + 1] = FIRST_CLASS_TOKEN + y
    x[:, 0] = START
    return x, y


probe_x, probe_y = make_batch(PROBES)

## The model

The model is a transformer with two layers of four attention heads each and width 64, about 104,000 parameters. It computes attention explicitly, so each layer returns its attention weights, and the output at the start token passes through a 3-D embedding layer before the classifier. AdamW trains it with a learning rate of 0.002.

In [ ]:
class Block(nn.Module):
    def __init__(self, width, heads):
        super().__init__()
        self.heads = heads
        self.norm1, self.norm2 = nn.LayerNorm(width), nn.LayerNorm(width)
        self.qkv, self.out = nn.Linear(width, 3 * width), nn.Linear(width, width)
        self.mlp = nn.Sequential(nn.Linear(width, 4 * width), nn.GELU(), nn.Linear(4 * width, width))

    def forward(self, h):
        b, t, w = h.shape
        q, k, v = self.qkv(self.norm1(h)).view(b, t, 3, self.heads, w // self.heads).permute(2, 0, 3, 1, 4)
        weights = (q @ k.transpose(-1, -2) / math.sqrt(w // self.heads)).softmax(-1)  # (batch, head, query, key)
        h = h + self.out((weights @ v).transpose(1, 2).reshape(b, t, w))
        return h + self.mlp(self.norm2(h)), weights


class TinyTransformer(nn.Module):
    def __init__(self, width=64, heads=4, layers=2):
        super().__init__()
        self.tokens = nn.Embedding(VOCAB, width)
        self.positions = nn.Parameter(0.02 * torch.randn(LENGTH, width))
        self.blocks = nn.ModuleList(Block(width, heads) for _ in range(layers))
        self.norm = nn.LayerNorm(width)
        self.embed = nn.Linear(width, 3)
        self.classify = nn.Linear(3, CLASSES)

    def forward(self, x):
        h = self.tokens(x) + self.positions
        attention = []
        for block in self.blocks:
            h, weights = block(h)
            attention.append(weights)
        z = self.embed(self.norm(h[:, 0]))
        return self.classify(z), z, torch.stack(attention)  # attention: (layer, batch, head, query, key)


model = TinyTransformer().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3)

## The landscape

The losses are taken on a fixed **landscape batch**, the first 128 of 512 fixed evaluation sequences. `Landscape` moves the 102,098 entries of the weight matrices and of the token and position embeddings. The biases and the LayerNorm gains and offsets, about 1,800 parameters, keep their current values.

Every 25 steps it keeps a snapshot of those weights on the GPU, at most 256 of them. When the buffer is full, it drops every second snapshot and doubles the interval, so the kept snapshots stay evenly spaced over the whole run. At each **refit** it finds the two directions in which the weights have moved most, the top two principal components of the snapshots taken around the current weights, and evaluates the loss on a 41 by 41 grid in the plane of those two directions, centred on the current weights. Nothing is read back to the CPU.

The scene `landscape` has three entities. `surface` is the grid as a mesh, with height and colour both the log of the loss: a vertex stands at half the log10 of the loss, clipped to the decades from 0.0001 to 100, so the plateau near the chance loss, ln 6 ≈ 1.79, and the losses after the fall, a thousandth or less, both stay in view. `path` is the optimizer's path in the plane, as a line, and `current` is the current point. The scene's root holds the box and the floor grid, logged once as static segments. The box spans -1 to 1 across and -2 to 1 in height in every mode, so the camera never has to move.

Three parameters and one action control the landscape from the panel below or a browser tab:

- `every`, the steps between refits, 500 by default;
- `span`, what the box's half-width covers: in the path's plane, 1.5 times `span` times the farthest snapshot's distance from the centre, so that at 1 the farthest snapshot sits two thirds of the way to the edge;
- `directions`, `path` for the path's principal components or `random` for two random directions;
- the action `surface now`, which refits at once, also while the run is paused.

A new `span` or `directions` takes effect at the next refit.

In [ ]:
EVAL = 512
eval_x, eval_y = make_batch(EVAL)

scene = caliper.scene("landscape", up="+z", units="")
every = caliper.param("every", 500, min=10, max=5000, help="steps between refits of the plane and surface")
span = caliper.param("span", 1.0, min=0.05, max=4.0, log=True, help="what the box's half-width covers")
directions = caliper.param("directions", "path", choices=["path", "random"],
                           help="the path's principal components, or random filter-normalised directions")
land = Landscape(model, eval_x[:128], eval_y[:128], grid=41, seed=0, snapshot_every=25, keep=256)
scene.lines("", land.frame, static=True)  # the box and the floor grid, as segments, once


def show_path(rate=10):
    scene.lines("path", land.path, strip_lengths=[land.path_length], colors=(235, 235, 235),
                rate=rate)
    scene.points("current", land.current, colors=(255, 80, 80), rate=rate)


def show_surface():
    positions, values, share = land.refit(model, span.value, directions.value)
    scene.mesh("surface", positions, land.triangles, values=values, cmap="viridis",
               vmin=-4.0, vmax=2.0)
    if share is not None:
        caliper.series(share, name="plane share")  # the path's variance the plane holds
    show_path(rate=0)  # the path in the new plane, at once


@caliper.action("surface now", help="refits the plane and the surface at the current weights")
def surface_now():
    if not land.can_fit(directions.value):
        return "the path's plane needs three snapshots, which exist at step 50"
    show_surface()
    return f"refitted at step {land.step}"

## The widgets

`caliper.controls()` is the panel of controls: the run state with the buttons Pause, Resume and Step, the three parameters and the action. Below it, `caliper.widget("landscape", "scene")` shows the scene. Drag to orbit, scroll to zoom, and double-click to fit the scene again. The small series below the scene is `plane share`, the share of the path's variance that the plane holds at each refit, which says how much of the path the picture shows.

In [ ]:
from ipywidgets import VBox

VBox([caliper.controls(),
      caliper.widget("landscape", "scene", height=480),
      caliper.widget("plane share", "series", height=120)])

## Training

Each step starts with `caliper.step()`, where a pause from a viewer holds the loop, new parameter values arrive and the action runs. The training step is the tutorial's. After it, `land.point()` runs the landscape batch through the model once, without gradients, for the height of the path's point, and keeps a snapshot every 25 steps. `land.due()` says when to refit: in the path's plane first at step 50, when three snapshots exist, and then at every multiple of `every`. Until step 50 the scene holds only the box and the floor grid. Between refits the loop sends the path and the current point at most 10 times a second.

**What the picture shows.** The surface is the loss over the two directions in which the weights have moved most, with the current weights at its centre. The path lies in the plane up to the part of its variance that the two directions miss, which `plane share` reports. Each refit computes new directions, so the axes change from one surface to the next and the path is drawn again in the new plane: one point for each kept snapshot, then one point per step until the next refit. The new directions are matched to the old ones so that the picture neither swaps its axes nor flips, which means that the first axis is not always the one that holds more of the path. The surface is the slice through the current weights, so earlier points of the path need not lie on it.

The loss usually stays on a plateau near the chance loss for one to a few thousand steps, while the path creeps over a broad, shallow surface and `plane share` may be low. Then the model learns to find the marker and the loss falls within a few hundred steps: the path runs downhill into a valley that each refit shows deeper, and the share rises as one direction dominates. Setting `every` to 50 when the loss starts to fall shows the valley open from one refit to the next.

In [ ]:
step = 0  # the run's step, continued by each call of train()


def train(steps):
    global step
    start = time.perf_counter()
    for _ in range(steps):
        caliper.step(step)  # pauses, new values and the action take effect here
        x, y = make_batch(BATCH)
        x[:PROBES] = probe_x  # the same probe sequences at the top of every batch
        y[:PROBES] = probe_y
        logits, z, attention = model(x)
        loss = F.cross_entropy(logits, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

        land.point(model, step)  # the path's point; a snapshot every 25 steps
        if land.due(step, every.value, directions.value):  # from step 50, then every N steps
            show_surface()
        elif land.ready:  # once a plane exists
            show_path()
        step += 1

    if land.ready:
        show_path(rate=0)  # the last step's point, past the rate limit
    final_loss = loss.item()  # one wait for the GPU, after the loop
    accuracy = (logits.argmax(1) == y).float().mean().item()
    seconds = time.perf_counter() - start
    print(f"{steps} steps in {seconds:.1f} s ({steps / seconds:.0f} steps/s), "
          f"loss {final_loss:.3f}, accuracy on the last batch {accuracy:.0%}, "
          f"{len(land.refits)} refits so far, the grid evaluated with {land.method}")

In [ ]:
train(STEPS)

## Random directions

Set `directions` to `random` in the controls panel, or run the next cell, which sets it from Python, refits at once as the action does, and trains a little longer.

In random directions the plane is spanned by two directions drawn once from a normal distribution with a fixed seed. Each row of each weight matrix and embedding in a direction is scaled to the norm of the same row of the weights at step 0, so a span of 1 perturbs every neuron's weights by about their own size. The directions and the anchor, the weights at step 0, stay fixed for the rest of the run, so successive surfaces are comparable and the path is never drawn again. The surface shows the local shape of the loss in two arbitrary directions.

The path barely moves in this plane. Two random directions in a space of about 102,000 dimensions are nearly orthogonal to any displacement of the weights, which projects onto each with about 0.3 % of its length, so the path reads as an almost vertical line at the centre, falling as the loss falls. That is why the path's plane is the default: only there does the path move across the surface.

In [ ]:
directions.set("random")
show_surface()  # refit now, as the action does
train(max(1, STEPS // 10))

## Closing

`server.close()` stops the server and ends the stream to every widget and browser tab.

In [ ]:
server.close()